# 05 · Splits & Class Weights

**Purpose.** Create the train / validation / test split once, leakage-free and as balanced as possible, then freeze it.

| | |
|---|---|
| **Reads** | `data/processed/manifest_clean.csv` |
| **Writes** | `data/splits/{train,val,test,manifest_split}.csv`, `data/splits/class_info.json` |
| **Run time (Colab T4)** | ~1 min |
| **Next notebook** | 06 · Dataset & Transforms |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

## 1. Choose the most balanced seed

`StratifiedGroupKFold` keeps every patient in one fold and balances classes approximately. How well
it balances depends on the random seed, so we try many seeds and keep the one whose val and test
class mix is closest to train's. A seed is rejected if any class has too few val/test images.

In [ ]:
import json
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd
from dermalens import splits as sp
from dermalens.labels import model_classes
from dermalens.utils import save_json

TARGET_CLASSES = model_classes(open_set=CONFIG["data"]["open_set"])   # includes "Other" in v3

data = pd.read_csv(paths.processed / "manifest_clean.csv", dtype={"phash": str})
settings = CONFIG["data"]

best_seed, seed_scores = sp.choose_split_seed(
    data, seeds=range(settings["split_seed_trials"]), n_folds=settings["split_folds"],
    min_per_class=settings["min_images_per_class_in_eval"])
seed_scores.to_csv(paths.audit / "split_seed_search.csv", index=False)
print("Chosen seed:", best_seed)
display(seed_scores.head(8))

## 2. Make the split and prove there is no leakage

In [ ]:
split = sp.make_grouped_split(data, seed=best_seed, n_folds=settings["split_folds"])
sp.check_no_leakage(split)                    # stops with an error if any patient is shared
print("No patient or file appears in two splits.")

counts = sp.split_summary(split)
display(counts)
shares = (pd.crosstab(split["label"], split["split"], normalize="columns") * 100).round(1)[["train", "val", "test"]]
display(shares)
print(f"Largest class-share drift from train: {sp.split_balance_score(split):.1f} percentage points")

## 3. Sources and skin-tone labels inside each split

In [ ]:
display(pd.crosstab(split["source"], split["split"])[["train", "val", "test"]])
display(split.groupby("split")["fst"].agg(with_fst="count", total="size").loc[["train", "val", "test"]])
shares.plot.bar(figsize=(9, 4), title="Class share per split (%)")
plt.tight_layout()
plt.savefig(paths.figures / "split_balance.png", dpi=150)
plt.show()

## 4. Class weights

`w_c = N / (C · n_c)` from the **training** split only. A class with half the average size gets
weight ≈ 2, so the loss pays twice as much attention to it.

In [ ]:
train = split[split["split"] == "train"]
class_weights = sp.compute_class_weights(train, TARGET_CLASSES)
display(pd.Series(class_weights, name="weight").to_frame())

## 5. Save and freeze
After this cell, nobody edits `data/splits/` without a team decision: every result depends on it.

In [ ]:
for name in ("train", "val", "test"):
    split[split["split"] == name].to_csv(paths.splits / f"{name}.csv", index=False)
split.to_csv(paths.splits / "manifest_split.csv", index=False)

class_info = {
    "classes": TARGET_CLASSES,
    "class_weights": class_weights,
    "split_seed": best_seed,
    "split_folds": settings["split_folds"],
    "images_per_split": split["split"].value_counts().to_dict(),
    "patients_per_split": split.groupby("split")["group_id"].nunique().to_dict(),
    "created": date.today().isoformat(),
    "open_set": CONFIG["data"]["open_set"],
    "note": "Healthy = MCSI (CC BY 4.0); Psoriasis = DermaCon-IN only; Other = non-target diseases.",
}
save_json(class_info, paths.splits / "class_info.json")
print(json.dumps(class_info, indent=1))